# Week 7 - Task 3: Model Evaluation & Interpretability

## Objective

This notebook evaluates a high-performing binary classification model using precision, recall, F1-score, confusion matrix, ROC-AUC, and SHAP explainability.

The project uses the Breast Cancer Wisconsin (Diagnostic) dataset from Scikit-Learn and a tuned Gradient Boosting classifier.

### Workflow
1. Load and split the dataset.
2. Train a strong classification model.
3. Generate precision, recall, F1, confusion matrix, and classification report.
4. Plot the ROC curve and calculate AUC.
5. Configure SHAP for global and local explanations.
6. Generate a SHAP summary plot and individual prediction explanations.
7. Interpret feature contributions in the context of the domain.

## Dataset

The **Breast Cancer Wisconsin (Diagnostic)** dataset is available through Scikit-Learn.

It contains:
- 569 observations
- 30 numerical features
- Binary target: malignant (0) or benign (1)

The features are computed from digitized images of fine needle aspirate (FNA) of breast masses. The model is used here for educational machine-learning evaluation and interpretability; it is not a medical diagnostic system.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score,
    roc_curve, roc_auc_score
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

ASSET_DIR = "../assets"
os.makedirs(ASSET_DIR, exist_ok=True)

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
display(X.head())

## 1. Train the Classification Model

Gradient Boosting is selected as the main model because it is a strong nonlinear classifier and works well with SHAP's tree-based explanation approach.

The configuration below is deliberately fixed so the notebook remains reproducible.

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

## 2. Classification Metrics

Precision answers: **Of the samples predicted as positive, how many were actually positive?**

Recall answers: **Of the actual positive samples, how many did the model find?**

F1-score is the harmonic mean of precision and recall and is useful when both error types matter.

The full classification report is also generated for both classes.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Score": [accuracy, precision, recall, f1]
})

display(metrics.round(4))
print(classification_report(
    y_test,
    y_pred,
    target_names=dataset.target_names
))

## 3. Confusion Matrix

The confusion matrix shows the number of correct and incorrect predictions for each class.

Because the target labels are `0 = malignant` and `1 = benign`, the matrix should be interpreted together with the class names rather than by treating the positive class as a medical recommendation.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=dataset.target_names
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format="d")
ax.set_title("Confusion Matrix - Gradient Boosting")
plt.tight_layout()
plt.savefig(os.path.join(ASSET_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 4. ROC Curve and AUC

The ROC curve evaluates the model over different classification thresholds.

AUC summarizes the ranking ability of the model across thresholds:
- 0.5 is approximately random ranking.
- 1.0 represents perfect ranking.

AUC should be interpreted alongside precision, recall, F1, and the confusion matrix.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"Gradient Boosting (AUC = {auc_score:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(ASSET_DIR, "roc_curve.png"), dpi=150)
plt.show()

print("ROC-AUC:", round(auc_score, 4))

## 5. SHAP Setup

SHAP (SHapley Additive exPlanations) estimates how individual features contribute to a prediction.

For tree-based models, `TreeExplainer` is an efficient choice.

If SHAP is not installed, install the dependencies from `requirements.txt`.

In [ ]:
import shap

print("SHAP version:", shap.__version__)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# SHAP versions can return either an array or a list for classifiers.
if isinstance(shap_values, list):
    shap_values_positive = shap_values[1]
else:
    shap_values_positive = shap_values

print("SHAP values shape:", np.asarray(shap_values_positive).shape)

## 6. Global Explanation - SHAP Summary Plot

The summary plot shows:
- which features have the greatest average impact,
- whether high or low feature values tend to push predictions in a direction,
- the distribution of feature contributions across samples.

For this binary classifier, the explanation focuses on the model's positive-class output.

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_positive,
    X_test,
    show=False
)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.savefig(
    os.path.join(ASSET_DIR, "shap_summary.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()

## 7. SHAP Bar Plot - Global Feature Importance

The bar summary ranks features by mean absolute SHAP value. This is a useful global measure of how strongly each feature contributes to predictions, without focusing on direction.

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values_positive,
    X_test,
    plot_type="bar",
    show=False
)
plt.title("Mean Absolute SHAP Feature Importance")
plt.tight_layout()
plt.savefig(
    os.path.join(ASSET_DIR, "shap_feature_importance.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()

## 8. Local Explanation - Force Plot

A force plot explains one individual prediction by showing features that push the prediction higher or lower relative to the model's baseline.

For compatibility with static files, the notebook also saves a waterfall plot, which is often easier to inspect inside GitHub.

In [ ]:
sample_index = 0

try:
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_values_positive[sample_index],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
            data=X_test.iloc[sample_index].values,
            feature_names=X_test.columns.tolist()
        ),
        show=False
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(ASSET_DIR, "shap_local_waterfall.png"),
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()
except Exception as exc:
    print("Waterfall plot could not be generated with this SHAP version:", exc)

## 9. Interactive Force Plot

The following cell creates an interactive SHAP force plot. Depending on the SHAP version, the exact rendering may differ.

The saved HTML file is useful when viewing the project locally in a Jupyter environment.

In [ ]:
try:
    expected_value = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, (list, np.ndarray))
        else explainer.expected_value
    )

    force = shap.force_plot(
        expected_value,
        shap_values_positive[sample_index],
        X_test.iloc[sample_index],
        matplotlib=False
    )
    shap.save_html(
        os.path.join(ASSET_DIR, "shap_force_plot.html"),
        force
    )
    print("Saved assets/shap_force_plot.html")
except Exception as exc:
    print("Interactive force plot could not be saved:", exc)

## 10. Top Features and Interpretation

The next cell creates a ranked table of mean absolute SHAP values.

In this dataset, features related to cell geometry, size, texture, and concavity commonly appear among influential variables. The exact ranking is model- and split-dependent, so the generated SHAP results should be used rather than assuming a fixed ranking.

In [ ]:
mean_abs_shap = np.abs(shap_values_positive).mean(axis=0)

importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Mean_Absolute_SHAP": mean_abs_shap
}).sort_values("Mean_Absolute_SHAP", ascending=False)

display(importance.head(15).round(6))
importance.to_csv(
    os.path.join(ASSET_DIR, "shap_feature_importance.csv"),
    index=False
)

## 11. Evaluation Summary

The following summary is saved for the final report.

Interpretation should emphasize that SHAP explains the behavior of the trained model; it does not prove causal relationships. Domain knowledge is useful for assessing whether the model's important features are plausible.

In [ ]:
summary = pd.DataFrame([{
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "ROC_AUC": auc_score
}])

display(summary.round(4))
summary.to_csv(
    os.path.join(ASSET_DIR, "evaluation_metrics.csv"),
    index=False
)

print("\nGenerated assets:")
for filename in sorted(os.listdir(ASSET_DIR)):
    print("-", filename)